Load Datasets

In [1]:
import pandas as pd
profiles_df = pd.read_csv(
    "../datasets/processed/climate_profiles.csv"
)
anomaly_df = pd.read_csv(
    "../datasets/processed/climate_anomalies.csv"
)
rainfall_df = pd.read_csv(
    "../datasets/processed/climate_rainfall_predictions.csv"
)
heatwave_df = pd.read_csv(
    "../datasets/processed/climate_heatwave_predictions.csv"
)
print(profiles_df.shape)
print(anomaly_df.shape)
print(rainfall_df.shape)
print(heatwave_df.shape)

(119398, 146)
(119398, 149)
(119398, 150)
(119398, 150)


Create Master Climate Intelligence Dataset

In [2]:
master_df = heatwave_df.copy()

master_df["rainfall_risk"] = rainfall_df["rainfall_risk"]

master_df["climate_profile"] = profiles_df["climate_profile"]

master_df["anomaly_status"] = anomaly_df["anomaly_status"]

print(master_df.shape)

print(
    master_df[
        ["climate_profile", "anomaly_status", "rainfall_risk", "heatwave_risk"]
    ].head()
)

(119398, 151)
  climate_profile anomaly_status rainfall_risk heatwave_risk
0        Moderate         Normal           Low       Warning
1        Moderate         Normal           Low       Warning
2        Moderate         Normal           Low       Warning
3        Moderate         Normal           Low       Warning
4        Moderate         Normal           Low       Warning


RISK SCORE MAPPINGS

In [3]:
rainfall_scores = {"Low": 10, "Medium": 40, "High": 80}
heatwave_scores = {"Safe": 10, "Warning": 50, "Critical": 90}

anomaly_scores = {"Normal": 0, "Anomaly": 100}

profile_scores = {
    "Moderate": 20,
    "Pollution-Prone": 50,
    "Flood-Prone": 70,
    "Extreme-Pollution": 90,
}

CLIMATE RISK SCORE

In [4]:
master_df["rainfall_score"] = master_df["rainfall_risk"].map(rainfall_scores)
master_df["heatwave_score"] = master_df["heatwave_risk"].map(heatwave_scores)

master_df["anomaly_score_final"] = master_df["anomaly_status"].map(anomaly_scores)

master_df["profile_score"] = master_df["climate_profile"].map(profile_scores)

master_df["climate_risk_score"] = (
    0.30 * master_df["rainfall_score"]
    + 0.30 * master_df["heatwave_score"]
    + 0.20 * master_df["profile_score"]
    + 0.20 * master_df["anomaly_score_final"]
)

master_df["climate_risk_score"] = master_df["climate_risk_score"].round(0)

print(master_df["climate_risk_score"].describe())

count    119398.000000
mean         18.155497
std           9.993843
min          10.000000
25%          10.000000
50%          16.000000
75%          22.000000
max          85.000000
Name: climate_risk_score, dtype: float64


RISK CATEGORY

In [5]:
def climate_risk_category(score):
    if score <= 16:
        return "Low"
    elif score <= 32:
        return "Medium"
    elif score <= 53:
        return "High"
    else:
        return "Critical"

master_df["climate_risk"] = master_df["climate_risk_score"].apply(climate_risk_category)

print(master_df["climate_risk"].value_counts())

print("\nPercentage Distribution")

print(round(master_df["climate_risk"].value_counts(normalize=True) * 100, 2))

climate_risk
Low         69094
Medium      40252
High         9475
Critical      577
Name: count, dtype: int64

Percentage Distribution
climate_risk
Low         57.87
Medium      33.71
High         7.94
Critical     0.48
Name: proportion, dtype: float64


SAVE FINAL DATASET

In [6]:
master_df.to_csv("../datasets/processed/climate_risk_intelligence.csv", index=False)

print("Climate Risk Intelligence Dataset Saved Successfully")

print(master_df.shape)

Climate Risk Intelligence Dataset Saved Successfully
(119398, 157)


In [7]:
master_df["climate_risk_score"].quantile(
    [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

0.25    10.0
0.50    16.0
0.75    22.0
0.90    32.0
0.95    41.0
0.99    53.0
Name: climate_risk_score, dtype: float64